<a href="https://colab.research.google.com/github/Sumit-0106/Ml-Repo/blob/main/LLM'S.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import torch
!pip install torch torchvision torchaudio
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

PyTorch version: 2.10.0+cpu
CUDA available: False


In [2]:
corpus = [
    "hello friends how are you",
    "the tea is very hot",
    "my name is Aarohi",
    "the roads of Delhi are busy",
    "it is raining in Mumbai",
    "the train is late again",
    "i love eating samosas and drinking tea",
    "holi is my favorite festival",
    "diwali brings lights and sweets",
    "india won the cricket match"
]


In [3]:
corpus = [s + "<END>" for s in corpus]
text = " ".join(corpus)
print(text)

hello friends how are you<END> the tea is very hot<END> my name is Aarohi<END> the roads of Delhi are busy<END> it is raining in Mumbai<END> the train is late again<END> i love eating samosas and drinking tea<END> holi is my favorite festival<END> diwali brings lights and sweets<END> india won the cricket match<END>


In [4]:
words = list(set(text.split()))
print(words)

['brings', 'my', 'samosas', 'tea', 'in', 'busy<END>', 'festival<END>', 'very', 'india', 'won', 'cricket', 'drinking', 'the', 'love', 'hot<END>', 'sweets<END>', 'Delhi', 'diwali', 'are', 'late', 'i', 'tea<END>', 'again<END>', 'name', 'match<END>', 'it', 'roads', 'friends', 'hello', 'how', 'is', 'eating', 'and', 'Aarohi<END>', 'favorite', 'raining', 'you<END>', 'holi', 'Mumbai<END>', 'train', 'of', 'lights']


In [5]:
word2idx ={w:i for i ,w in enumerate(words)}
print(word2idx)

{'brings': 0, 'my': 1, 'samosas': 2, 'tea': 3, 'in': 4, 'busy<END>': 5, 'festival<END>': 6, 'very': 7, 'india': 8, 'won': 9, 'cricket': 10, 'drinking': 11, 'the': 12, 'love': 13, 'hot<END>': 14, 'sweets<END>': 15, 'Delhi': 16, 'diwali': 17, 'are': 18, 'late': 19, 'i': 20, 'tea<END>': 21, 'again<END>': 22, 'name': 23, 'match<END>': 24, 'it': 25, 'roads': 26, 'friends': 27, 'hello': 28, 'how': 29, 'is': 30, 'eating': 31, 'and': 32, 'Aarohi<END>': 33, 'favorite': 34, 'raining': 35, 'you<END>': 36, 'holi': 37, 'Mumbai<END>': 38, 'train': 39, 'of': 40, 'lights': 41}


In [7]:
data = torch.tensor([word2idx[w] for w in text.split()])
print(data)
print(len(data))

tensor([28, 27, 29, 18, 36, 12,  3, 30,  7, 14,  1, 23, 30, 33, 12, 26, 40, 16,
        18,  5, 25, 30, 35,  4, 38, 12, 39, 30, 19, 22, 20, 13, 31,  2, 32, 11,
        21, 37, 30,  1, 34,  6, 17,  0, 41, 32, 15,  8,  9, 12, 10, 24])
52


In [21]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class Head(nn.Module):
    def __init__(self, head_size, embedding_dim, block_size):
        super().__init__()
        self.key = nn.Linear(embedding_dim, head_size, bias=False)
        self.query = nn.Linear(embedding_dim, head_size, bias=False)
        self.value = nn.Linear(embedding_dim, head_size, bias=False)
        self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))
        self.dropout = nn.Dropout(0.1)

    def forward(self, x):
        B, T, C = x.shape
        k = self.key(x)
        q = self.query(x)
        v = self.value(x)

        wei = q @ k.transpose(-2, -1) * C**-0.5
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float('-inf'))
        wei = F.softmax(wei, dim=-1)
        wei = self.dropout(wei)

        out = wei @ v
        return out

class MultiHeadAttention(nn.Module):
    def __init__(self, embedding_dim, block_size, n_heads):
        super().__init__()
        head_size = embedding_dim // n_heads
        self.heads = nn.ModuleList([Head(head_size, embedding_dim, block_size) for _ in range(n_heads)])
        self.proj = nn.Linear(embedding_dim, embedding_dim)
        self.dropout = nn.Dropout(0.1)

    def forward(self, x):
        out = torch.cat([h(x) for h in self.heads], dim=-1)
        out = self.proj(out)
        out = self.dropout(out)
        return out

class FeedForward(nn.Module):
    def __init__(self, embedding_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(embedding_dim, 4 * embedding_dim),
            nn.ReLU(),
            nn.Linear(4 * embedding_dim, embedding_dim),
            nn.Dropout(0.1)
        )

    def forward(self, x):
        return self.net(x)

class Block(nn.Module):
    def __init__(self, embedding_dim, block_size, n_heads):
        super().__init__()
        self.sa = MultiHeadAttention(embedding_dim, block_size, n_heads)
        self.ffwd = FeedForward(embedding_dim)
        self.ln1 = nn.LayerNorm(embedding_dim)
        self.ln2 = nn.LayerNorm(embedding_dim)

    def forward(self, x):
        x = x + self.sa(self.ln1(x))
        x = x + self.ffwd(self.ln2(x))
        return x

In [10]:
block_size = 6
embedding_dim = 32
n_heads = 2
n_layers = 2
lr = 1e-3
epochs = 1500


def get_batch(batch_size = 16):
  ix = torch.randint(len(data) - block_size, (batch_size,))
  x = torch.stack([data[i:i+block_size] for i in ix])
  y = torch.stack([data[i+1:i+block_size+1] for i in ix])
  return x, y

In [22]:
class TinyGPT(nn.Module):
  def __init__(self): # Corrected __init__
    super().__init__() # Corrected super().__init__()
    self.token_embedding = nn.Embedding(len(words), embedding_dim)

    self.position_embedding = nn.Embedding(block_size, embedding_dim)
    # Corrected Block instantiation to pass block_size
    self.blocks = nn.Sequential(*[Block(embedding_dim, block_size, n_heads) for _ in range(n_layers)])
    self.ln_f = nn.LayerNorm(embedding_dim)
    self.lm_head = nn.Linear(embedding_dim, len(words))

  def forward(self, idx, targets=None):
        B, T = idx.shape
        tok_emb = self.token_embedding(idx)

        pos_emb = self.position_embedding(torch.arange(T, device=idx.device))
        x = tok_emb + pos_emb
        x = self.blocks(x)
        x = self.ln_f(x)
        logits = self.lm_head(x) # Corrected self.head to self.lm_head
        loss = None
        if targets is not None:
            B, T, C = logits.shape
            loss = F.cross_entropy(logits.view(B*T, C), targets.view(B*T))
        return logits, loss

  def generate(self, idx, max_new_tokens):
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -block_size:]
            logits, _ = self(idx_cond)
            logits = logits[:, -1, :]
            probs = F.softmax(logits, dim=-1)
            next_idx = torch.multinomial(probs, 1)
            idx = torch.cat((idx, next_idx), dim=1)
        return idx

In [15]:
def forward(self, idx, targets=None):
        B, T = idx.shape
        tok_emb = self.token_embedding(idx)

        pos_emb = self.position_embedding(torch.arange(T, device=idx.device))
        x = tok_emb + pos_emb
        x = self.blocks(x)
        x = self.ln_f(x)
        logits = self.head(x)
        loss = None
        if targets is not None:
            B, T, C = logits.shape
            loss = F.cross_entropy(logits.view(B*T, C), targets.view(B*T))
        return logits, loss

In [18]:
def generate(self, idx, max_new_tokens):
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -block_size:]
            logits, _ = self(idx_cond)
            logits = logits[:, -1, :]
            probs = F.softmax(logits, dim=-1)
            next_idx = torch.multinomial(probs, 1)
            idx = torch.cat((idx, next_idx), dim=1)
        return idx



In [23]:
idx2word = {i:w for w,i in word2idx.items()}

model = TinyGPT()
optimizer = torch.optim.AdamW(model.parameters(), lr=lr)

for step in range(epochs):
    xb, yb = get_batch()
    logits, loss = model(xb, yb)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    if step % 300 == 0:
        print(f"Step {step}, loss={loss.item():.4f}")



context = torch.tensor([[word2idx["hello"]]], dtype=torch.long)
out = model.generate(context, max_new_tokens=15)

print("\nGenerated text:\n")
print(" ".join(idx2word[int(i)] for i in out[0]))

Step 0, loss=4.0370
Step 300, loss=0.2134
Step 600, loss=0.1174
Step 900, loss=0.0402
Step 1200, loss=0.1365

Generated text:

hello friends how are you<END> the tea is very lights and sweets<END> india won the cricket
